# 🏠 DOMIAN — Entrenamiento Adaptive Classifier v3
## WiFi CSI Sensing · 3 clases · GPU T4

**Clases:**
- `absent` → habitación vacía
- `present_still` → persona quieta
- `present_moving` → persona moviéndose

**Objetivo:** Subir de 72.3% a 80%+ accuracy con la clase absent incluida

---
**Antes de ejecutar:** Cambia el tipo de entorno de ejecución a **T4 GPU**

`Entorno de ejecución → Cambiar tipo → T4 GPU → Guardar`

## Paso 1 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado')

## Paso 2 — Instalar dependencias

In [ ]:
!pip install torch numpy scikit-learn tqdm -q
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ PyTorch {torch.__version__} — usando: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

## Paso 3 — Descomprimir grabaciones

Asegúrate de que `grabaciones_domian.zip` esté en `Google Drive/DOMIAN/`

In [ ]:
import os

ZIP_PATH = '/content/drive/MyDrive/DOMIAN/grabaciones_domian.zip'
OUT_DIR  = '/content/recordings'

if not os.path.exists(ZIP_PATH):
    print('❌ No se encontró grabaciones_domian.zip en Google Drive/DOMIAN/')
    print('   Sube el archivo ZIP desde la Mac primero')
else:
    print('📦 Descomprimiendo...')
    !unzip -q "{ZIP_PATH}" -d "{OUT_DIR}"
    archivos = os.listdir(OUT_DIR)
    print(f'✅ {len(archivos)} archivos encontrados:')
    for f in sorted(archivos):
        size = os.path.getsize(f'{OUT_DIR}/{f}') / 1024 / 1024
        print(f'   {f} — {size:.1f} MB')

## Paso 4 — Cargar datos con features de 15 dimensiones

Compatible con `N_FEATURES = 15` del servidor RuView

In [ ]:
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm

def extraer_features_15(obj):
    """Extrae vector de 15 features compatible con N_FEATURES=15 del servidor"""
    resultados = []
    for nf in obj.get('node_features', []):
        f   = nf.get('features', {})
        clf = nf.get('classification', {})
        row = [
            float(f.get('mean_rssi', 0)),
            float(f.get('variance', 0)),
            float(f.get('motion_band_power', 0)),
            float(f.get('breathing_band_power', 0)),
            float(f.get('dominant_freq_hz', 0)),
            float(f.get('change_points', 0)),
            float(f.get('spectral_power', 0)),
            float(nf.get('rssi_dbm', 0)),
            float(nf.get('last_seen_ms', 0)),
            float(nf.get('frame_rate_hz', 0)),
            1.0 if nf.get('stale') else 0.0,
            1.0 if clf.get('presence') else 0.0,
            float(clf.get('confidence', 0)),
            float(nf.get('node_id', 0)),
            0.0  # padding
        ]
        resultados.append(row)
    return resultados

# Mapeo de prefijos a labels
LABEL_MAP = {
    'train_absent':         0,
    'train_present_still':  1,
    'train_present_moving': 2,
}
CLASE_NOMBRES = ['absent', 'present_still', 'present_moving']

def cargar_grabacion(path, label):
    frames = []
    with open(path, 'r') as f:
        lineas = f.readlines()
    for linea in tqdm(lineas, desc=Path(path).name[:40], leave=False):
        try:
            obj = json.loads(linea.strip())
            for row in extraer_features_15(obj):
                frames.append((np.array(row, dtype=np.float32), label))
        except:
            continue
    return frames

# Cargar todos los archivos
datos = []
archivos = sorted(os.listdir(OUT_DIR))

print('Cargando grabaciones...')
for archivo in archivos:
    label = None
    for prefijo, lbl in LABEL_MAP.items():
        if archivo.startswith(prefijo):
            label = lbl
            break
    if label is None:
        print(f'  ⚠️  {archivo} — prefijo no reconocido, ignorado')
        continue
    frames = cargar_grabacion(f'{OUT_DIR}/{archivo}', label)
    print(f'  ✅ {archivo}: {len(frames):,} frames → clase {CLASE_NOMBRES[label]}')
    datos += frames

print(f'\nTotal: {len(datos):,} frames')
for i, nombre in enumerate(CLASE_NOMBRES):
    count = sum(1 for _, l in datos if l == i)
    print(f'  {nombre}: {count:,} frames ({100*count/len(datos):.1f}%)')

## Paso 5 — Preparar datasets y normalizar

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch

X = np.array([f for f, _ in datos])
y = np.array([l for _, l in datos])

# Split 80/20
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalizar (guardar stats para exportar)
mean15 = X_train.mean(axis=0)
std15  = X_train.std(axis=0) + 1e-8

X_train_n = (X_train - mean15) / std15
X_val_n   = (X_val   - mean15) / std15

# DataLoaders
train_dl = DataLoader(
    TensorDataset(torch.FloatTensor(X_train_n), torch.LongTensor(y_train)),
    batch_size=512, shuffle=True, num_workers=2
)
val_dl = DataLoader(
    TensorDataset(torch.FloatTensor(X_val_n), torch.LongTensor(y_val)),
    batch_size=512
)

print(f'Train: {len(X_train):,} frames')
print(f'Val:   {len(X_val):,} frames')
print(f'\nDistribución val:')
for i, nombre in enumerate(CLASE_NOMBRES):
    count = (y_val == i).sum()
    print(f'  {nombre}: {count:,}')

## Paso 6 — Definir modelo

In [ ]:
import torch.nn as nn

class DomianCSIClassifier(nn.Module):
    """Clasificador CSI 15-dim → 3 clases para servidor RuView"""
    def __init__(self, input_dim=15, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)

    def get_linear_weights(self):
        """Proyecta pesos al espacio de 15 features para exportar al servidor"""
        w1 = self.net[0].weight.detach().cpu().numpy()   # 128x15
        w2 = self.net[4].weight.detach().cpu().numpy()   # 64x128
        w3 = self.net[8].weight.detach().cpu().numpy()   # 32x64
        w4 = self.net[9].weight.detach().cpu().numpy()   # 3x32
        b4 = self.net[9].bias.detach().cpu().numpy()     # 3
        w_combined = w4 @ w3 @ w2 @ w1  # 3x15
        weights = []
        for i in range(3):
            weights.append(w_combined[i].tolist() + [float(b4[i])])
        return weights

model = DomianCSIClassifier(num_classes=len(CLASE_NOMBRES)).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'✅ Modelo creado — {total_params:,} parámetros')
print(f'   Arquitectura: 15 → 128 → 64 → 32 → {len(CLASE_NOMBRES)} clases')

## Paso 7 — Entrenar

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS   = 40
LR       = 0.001
MODEL_PT = '/content/domian_model_v3.pt'

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

best_acc  = 0.0
historial = []

print(f'Entrenando {EPOCHS} épocas en {device}...')
print(f'{"Epoch":6} {"Train Loss":12} {"Val Acc":10} {"Best":8} {"LR":10}')
print('-' * 50)

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    # Val
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total   += len(yb)
    acc = correct / total

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), MODEL_PT)
        saved = '💾'
    else:
        saved = ''

    historial.append({'epoch': epoch, 'loss': total_loss/len(train_dl), 'acc': acc})

    if epoch % 5 == 0 or epoch == 1:
        lr_now = scheduler.get_last_lr()[0]
        print(f'{epoch:6} {total_loss/len(train_dl):12.4f} {acc:10.4f} {best_acc:8.4f} {lr_now:10.6f} {saved}')

print(f'\n✅ Entrenamiento completo')
print(f'   Mejor accuracy: {best_acc:.4f} ({best_acc*100:.1f}%)')

## Paso 8 — Evaluación por clase

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Cargar mejor modelo
model.load_state_dict(torch.load(MODEL_PT))
model.eval()

all_preds = []
all_true  = []

with torch.no_grad():
    for xb, yb in val_dl:
        xb = xb.to(device)
        preds = model(xb).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(yb.numpy())

print('Classification Report:')
print(classification_report(all_true, all_preds, target_names=CLASE_NOMBRES))

print('Confusion Matrix:')
cm = confusion_matrix(all_true, all_preds)
print(f'                 {"  ".join(CLASE_NOMBRES)}')
for i, row in enumerate(cm):
    print(f'{CLASE_NOMBRES[i]:20} {row}')

## Paso 9 — Exportar en formato RuView

Genera `adaptive_model.json` compatible con el servidor Rust

In [ ]:
import json

# Cargar mejor modelo
model.load_state_dict(torch.load(MODEL_PT))
model.eval()

# Pesos proyectados [3 clases x 16 (15 features + 1 bias)]
weights = model.get_linear_weights()
print(f'Weights shape: {len(weights)} x {len(weights[0])}')

# Stats por clase
class_stats = []
for label_id, nombre in enumerate(CLASE_NOMBRES):
    mask   = y == label_id
    X_cls  = X[mask]
    class_stats.append({
        'label':  nombre,
        'count':  int(mask.sum()),
        'mean':   X_cls.mean(axis=0).tolist(),
        'stddev': X_cls.std(axis=0).tolist()
    })
    print(f'  {nombre}: {mask.sum():,} frames')

modelo_ruview = {
    'class_stats':       class_stats,
    'weights':           weights,
    'global_mean':       mean15.tolist(),
    'global_std':        std15.tolist(),
    'trained_frames':    len(X),
    'training_accuracy': best_acc,
    'version':           1,
    'class_names':       CLASE_NOMBRES
}

# Guardar localmente y en Drive
LOCAL_PATH = '/content/adaptive_model_v3.json'
DRIVE_PATH = '/content/drive/MyDrive/DOMIAN/modelo_v3/adaptive_model_v3.json'

os.makedirs('/content/drive/MyDrive/DOMIAN/modelo_v3', exist_ok=True)

with open(LOCAL_PATH, 'w') as f:
    json.dump(modelo_ruview, f)
with open(DRIVE_PATH, 'w') as f:
    json.dump(modelo_ruview, f)

print(f'\n✅ Modelo exportado')
print(f'   Local:  {LOCAL_PATH}')
print(f'   Drive:  {DRIVE_PATH}')
print(f'   Accuracy: {best_acc:.4f} ({best_acc*100:.1f}%)')

## Paso 10 — Descargar e instalar en servidor

Descarga el archivo y en la Mac ejecuta:

In [ ]:
from google.colab import files
files.download(LOCAL_PATH)
print('✅ Descargando adaptive_model_v3.json...')
print()
print('En la Mac ejecuta:')
print('  cp ~/Downloads/adaptive_model_v3.json \\')
print('     ~/Documents/RuView/v2/data/adaptive_model.json')
print()
print('Luego reinicia el servidor y verifica:')
print(f'  Loaded adaptive classifier: {len(X):,} frames, {best_acc*100:.1f}% accuracy')

## (Opcional) Paso 11 — Verificar el modelo localmente

In [ ]:
# Prueba de inferencia con un frame de ejemplo
import numpy as np
import torch

# Frame de prueba (valores típicos de habitación vacía)
frame_vacio   = np.array([0, 2.1, 1.5, 0.8, 0.3, 0, 5.2, -75, 50, 10, 0, 0, 0.1, 2, 0], dtype=np.float32)
frame_quieto  = np.array([-65, 19.8, 33.9, 28.1, 1.2, 5, 118.7, -65, 30, 10, 0, 1, 0.33, 2, 0], dtype=np.float32)
frame_moviendo= np.array([-62, 35.2, 58.4, 45.3, 2.1, 12, 145.2, -62, 25, 10, 0, 1, 0.85, 2, 0], dtype=np.float32)

model.eval()
for nombre_frame, frame in [('vacío', frame_vacio), ('quieto', frame_quieto), ('moviéndose', frame_moviendo)]:
    x = torch.FloatTensor((frame - mean15) / std15).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
    pred = CLASE_NOMBRES[probs.argmax()]
    print(f'Frame {nombre_frame:12} → {pred:20} (conf: {probs.max():.2f})')
    for i, c in enumerate(CLASE_NOMBRES):
        print(f'   {c:20}: {probs[i]:.3f}')